# Module 02: Exam Generation

Generate curriculum-grounded assessment questions with the reusable scripts in `scripts/generation.py`. This notebook loads the existing Chroma collection; it does not create, clear, or rebuild a vector store.

## 1. Project Setup

Find the repository root from either the project directory or `notebooks/`, import the shared configuration, and display only safe configuration details. Credentials are never displayed.

In [ ]:
import importlib
import sys
from pathlib import Path

candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
ROOT_DIR = next((path for path in candidates if (path / "scripts" / "config.py").is_file()), None)
if ROOT_DIR is None:
    raise FileNotFoundError("Could not find the project root containing scripts/config.py")
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from scripts.config import (
    ENV_FILE,
    GEMINI_API_KEY,
    HF_EMBEDDING_MODEL_NAME,
    LLM_MODEL_NAME,
    LLM_PROVIDER,
    OUTPUTS_DIR,
    VECTOR_DB_PATH,
)
import scripts.generation as generation_module
generation_module = importlib.reload(generation_module)
generate_exam = generation_module.generate_exam
list_curriculum_lessons = generation_module.list_curriculum_lessons
resolve_vectorstore_dir = generation_module.resolve_vectorstore_dir
validate_configuration = generation_module.validate_configuration
validate_exam = generation_module.validate_exam

configuration_ok, missing_configuration, configured_model = validate_configuration()
print(f"Project root: {ROOT_DIR}")
print(f"Runtime .env path: {ENV_FILE} (exists: {ENV_FILE.is_file()})")
print(f"LLM provider: {LLM_PROVIDER}")
print(f"Gemini API key configured: {'yes' if GEMINI_API_KEY else 'no'}")
print(f"Configured model: {configured_model}")
if not configuration_ok:
    raise RuntimeError(f"Missing configuration: {', '.join(missing_configuration)}")

## 2. Load Curriculum Resources

Use the Hugging Face sentence-transformer configured for the indexed collection. The collection is opened read-only for this workflow; its path, name, and current chunk count are shown below.

In [ ]:
from scripts.ingestion import get_embeddings, load_vectorstore

COLLECTION_NAME = "curriculum"
VECTOR_STORE_PATH = resolve_vectorstore_dir()
if not VECTOR_STORE_PATH.is_dir():
    raise FileNotFoundError(f"Existing curriculum vector store not found: {VECTOR_STORE_PATH}")

embeddings = get_embeddings(model_name=HF_EMBEDDING_MODEL_NAME)
vector_store = load_vectorstore(
    embeddings=embeddings,
    persist_dir=VECTOR_STORE_PATH,
    collection_name=COLLECTION_NAME,
)
chunk_count = vector_store._collection.count()
print(f"Embedding model: {HF_EMBEDDING_MODEL_NAME}")
print(f"Vector store: {VECTOR_STORE_PATH}")
print(f"Collection: {COLLECTION_NAME}")
print(f"Chunks available: {chunk_count}")

## 3. Select a Lesson and Set Exam Parameters

Choose a lesson ID from the list below, then set the requested question count (any positive number), question types, difficulty, and Bloom's taxonomy level. The default type mix includes MCQ, short-answer, and essay/open-ended questions. Each question requires one request to the configured model provider.

In [ ]:
LESSONS = list_curriculum_lessons(vector_store)
print("Available lessons (select a lesson_id below):")
for lesson in LESSONS:
    print(f"  {lesson['lesson_id']}: {lesson['lesson_title']} | chapter {lesson['chapter_id']} | {lesson['chunk_count']} chunks")

LESSON_ID = "1-3"  # Choose one lesson_id from the list above
SELECTED_LESSON = next((lesson for lesson in LESSONS if lesson["lesson_id"] == LESSON_ID), None)
if SELECTED_LESSON is None:
    raise ValueError(f"Unknown lesson_id {LESSON_ID!r}; choose an ID from the list above.")
TOPIC = SELECTED_LESSON["lesson_title"]
QUESTION_COUNT = 1
QUESTION_TYPES = ["mcq"]
DIFFICULTY = "medium"  # easy, medium, hard
BLOOM_LEVEL = "apply"  # remember, understand, apply, analyze, evaluate, create
RETRIEVAL_COUNT = max(8, QUESTION_COUNT)

if QUESTION_COUNT > SELECTED_LESSON["chunk_count"]:
    print(f"WARNING: requesting {QUESTION_COUNT} questions from only "
          f"{SELECTED_LESSON['chunk_count']} lesson chunks.")

print({
    "lesson_id": LESSON_ID,
    "lesson_title": TOPIC,
    "topic": TOPIC,
    "question_count": QUESTION_COUNT,
    "question_types": QUESTION_TYPES,
    "difficulty": DIFFICULTY,
    "bloom_level": BLOOM_LEVEL,
})

## 4. Retrieve Curriculum Context

Retrieve relevant chunks before asking the model to generate anything. The notebook stops if the existing collection returns no context, and shows source/chunk metadata with a short passage preview.

In [ ]:
from scripts.ingestion import retrieve

context_docs = retrieve(
    vector_store,
    query=TOPIC,
    k=RETRIEVAL_COUNT,
    lesson_id=LESSON_ID,
)
if not context_docs:
    raise ValueError(f"No curriculum context found for topic: {TOPIC!r}; exam generation stopped.")

for index, document in enumerate(context_docs, start=1):
    print(f"Passage {index}: {document.metadata}")
    print(document.page_content[:700].strip())
    print("-" * 72)

## 5. Generate the Exam

The shared generation function reuses retrieved curriculum passages, requests structured JSON from the configured Gemini or Ollama model, validates each question, and saves only a validated exam. MCQs receive a separate independent answer-verification model call (one extra LLM call per MCQ). Source-reference errors are retried, and invalid questions fail explicitly.

In [ ]:
exam, saved_exam_path = generate_exam(
    topic=TOPIC,
    question_types=QUESTION_TYPES,
    number_of_questions=QUESTION_COUNT,
    difficulty=DIFFICULTY,
    bloom_level=BLOOM_LEVEL,
    persist_dir=VECTOR_STORE_PATH,
    collection_name=COLLECTION_NAME,
    output_dir=OUTPUTS_DIR,
    context_docs=context_docs,
    lesson_id=LESSON_ID,
)
if len(exam["questions"]) != QUESTION_COUNT:
    raise ValueError(
        f"Requested {QUESTION_COUNT} questions but generated "
        f"{len(exam['questions'])}."
    )
print(f"Generated {exam['question_count']} question(s) with {exam['model']}.")
print(f"Warnings: {exam.get('warnings', [])}")

## 6. Validate Results

Validate the complete exam and display all question fields, including options, answer, explanation, rubric, and source references. Any validation failure raises an error rather than appearing as success.

In [ ]:
validate_exam(exam)
print("Validation passed.")
for question in exam["questions"]:
    print(f"\n{question['question_id']} [{question['question_type']}] {question['question']}")
    if question["question_type"] == "mcq":
        for option_label, option in zip("ABCD", question["options"]):
            if option.lstrip().upper().startswith(f"{option_label} "):
                print(f"  {option}")
            else:
                print(f"  {option_label}. {option}")
        print(f"Correct answer: {question['correct_answer']}")
    print(f"Reference answer: {question['reference_answer']}")
    print(f"Explanation: {question['explanation']}")
    print(f"Bloom rationale: {question['bloom_rationale']}")
    print(f"Rubric (max {question['max_score']} points): {question['rubric']}")
    print(f"Sources: {question['source_refs']}")

## 7. Save Results

The generation function has already saved the validated result using an exclusive create operation. Confirm that the unique output file exists and can be read as JSON; previous exam files are never replaced.

In [ ]:
import json

if not saved_exam_path.is_file():
    raise FileNotFoundError(f"Expected saved exam file is missing: {saved_exam_path}")
saved_exam = json.loads(saved_exam_path.read_text(encoding="utf-8"))
validate_exam(saved_exam)
print(f"Validated exam saved to: {saved_exam_path}")

## 8. Error Handling and Summary

Common blockers include a missing `GEMINI_API_KEY` or model setting, a missing vector-store collection, a model unavailable to the Gemini API key/project, an unavailable or mismatched embedding model, no relevant retrieved chunks, malformed model JSON, or Gemini authentication, quota, and rate-limit errors. If the model is unavailable, verify the exact model name and its availability for your Google AI Studio project. For permission errors, verify the key and API access; for quota errors, check the project's Gemini API limits and billing. The scripts raise a specific error for these conditions and do not save an invalid exam.

In [ ]:
print("Run summary")
print(f"  Topic: {TOPIC}")
print(f"  Requested questions: {QUESTION_COUNT}")
print(f"  Generated questions: {len(exam['questions'])}")
print(f"  Unique question texts: {len({q['question'].strip().casefold() for q in exam['questions']})}")
print(f"  Unique cited chunk IDs: {len({ref['chunk_id'] for q in exam['questions'] for ref in q.get('source_refs', []) if ref.get('chunk_id')})}")
print(f"  Retrieved passages: {len(context_docs)}")
print(f"  Warnings: {exam.get('warnings', [])}")
print(f"  Total points: {exam['total_points']}")
print(f"  Output: {saved_exam_path}")
